# wsproto smoke test

Sans-IO WebSocket protocol implementation (RFC 6455 + RFC 7692 permessage-deflate).

In [ ]:
# Example 1: Client handshake — generate the opening HTTP Upgrade request bytes.
from wsproto import WSConnection, ConnectionType
from wsproto.events import Request

client = WSConnection(ConnectionType.CLIENT)
request = Request(host="example.com", target="/chat")
data = client.send(request)

print(f"connection type: {ConnectionType.CLIENT}")
print(f"handshake bytes (first 120): {data[:120]!r}")

assert b"GET /chat HTTP/1.1" in data
assert b"Upgrade: WebSocket" in data or b"Upgrade: websocket" in data
assert b"Sec-WebSocket-Key" in data
assert b"Sec-WebSocket-Version: 13" in data

In [ ]:
# Example 2: Server side — feed the client's opening bytes and observe a Request event.
from wsproto import WSConnection, ConnectionType
from wsproto.events import Request, AcceptConnection

client = WSConnection(ConnectionType.CLIENT)
client_bytes = client.send(Request(host="example.com", target="/ws"))

server = WSConnection(ConnectionType.SERVER)
server.receive_data(client_bytes)

events = list(server.events())
print(f"server saw {len(events)} event(s): {[type(e).__name__ for e in events]}")

req_events = [e for e in events if isinstance(e, Request)]
assert len(req_events) == 1
assert req_events[0].target == "/ws"
assert req_events[0].host == "example.com"

In [ ]:
# Example 3: Full handshake + text message round trip (client -> server).
from wsproto import WSConnection, ConnectionType
from wsproto.events import Request, AcceptConnection, TextMessage

client = WSConnection(ConnectionType.CLIENT)
server = WSConnection(ConnectionType.SERVER)

server.receive_data(client.send(Request(host="h", target="/")))
list(server.events())  # drain Request
client.receive_data(server.send(AcceptConnection()))
list(client.events())  # drain AcceptConnection

server.receive_data(client.send(TextMessage(data="hello server")))
msgs = [e for e in server.events() if isinstance(e, TextMessage)]

print(f"server received: {[m.data for m in msgs]}")
assert msgs and msgs[0].data == "hello server"

In [ ]:
# Example 4: Binary message + Ping/Pong control frames round trip.
from wsproto import WSConnection, ConnectionType
from wsproto.events import Request, AcceptConnection, BytesMessage, Ping, Pong

client = WSConnection(ConnectionType.CLIENT)
server = WSConnection(ConnectionType.SERVER)
server.receive_data(client.send(Request(host="h", target="/"))); list(server.events())
client.receive_data(server.send(AcceptConnection())); list(client.events())

server.receive_data(client.send(BytesMessage(data=b"\x00\x01\x02")))
server.receive_data(client.send(Ping(payload=b"ping-me")))

events = list(server.events())
kinds = [type(e).__name__ for e in events]
print(f"server events: {kinds}")

assert any(isinstance(e, BytesMessage) and e.data == b"\x00\x01\x02" for e in events)
assert any(isinstance(e, Ping) and e.payload == b"ping-me" for e in events)

In [ ]:
# Example 5: Connection state transitions + clean close handshake.
from wsproto import WSConnection, ConnectionType
from wsproto.connection import ConnectionState
from wsproto.events import Request, AcceptConnection, CloseConnection

client = WSConnection(ConnectionType.CLIENT)
server = WSConnection(ConnectionType.SERVER)

server.receive_data(client.send(Request(host="h", target="/"))); list(server.events())
client.receive_data(server.send(AcceptConnection())); list(client.events())

print(f"after handshake: client={client.state}, server={server.state}")
assert client.state == ConnectionState.OPEN
assert server.state == ConnectionState.OPEN

server.receive_data(client.send(CloseConnection(code=1000, reason="bye")))
close_evts = [e for e in server.events() if isinstance(e, CloseConnection)]
client.receive_data(server.send(close_evts[0].response()))
list(client.events())

print(f"after close: client={client.state}, server={server.state}")
assert close_evts[0].code == 1000 and close_evts[0].reason == "bye"
assert client.state == ConnectionState.CLOSED